In [1]:
import numpy as np
import matplotlib.pyplot as plt

import os
import pandas as pd
import math
from sklearn.svm import SVC
import seaborn as sns
import pickle
import pingouin as pg
import glob


In [ ]:
# set path and load data

basedir = '/path/to/pain-induced-manifold-changes'
datdir = os.path.join(basedir, 'data')

file = os.path.join(datdir, 'pain-induced-manifold-changes_wholepaintrial_data_for_heatlevel_classification.pickle')


with open(file, 'rb') as f:
    all_sub_dat = pickle.load(f)

sub_ids = list(all_sub_dat.keys())

# set conditions / ROIs
condition = [0,1,2] # 0 - warmth; 1 - low heat; 2 - high heat
rois = ['V1', 'V2', 'hV4', 'LO']

In [ ]:
# run SVM


linear_SVM = SVC(kernel='linear')

all_sub_acc = pd.DataFrame({'sub':[], 'roi':[], 'cond1':[], 'cond2':[], 'mean_acc':[]})

for sub in sub_ids:    
    print(sub)
    for roi in rois:
                
        data = all_sub_dat[sub][roi]['data']
        attr_condition = all_sub_dat[sub][roi]['attr_condition']
        attr_runnum = all_sub_dat[sub][roi]['attr_runnum']    
        
        
        runs = np.unique(all_sub_dat[sub][roi]['attr_runnum'])
        for cond1 in condition:
            for cond2 in condition:
                
                if cond2 > cond1:
                
                    data_cond = data[(attr_condition == cond1) | (attr_condition == cond2)]
                    attr_condition_temp = attr_condition[(attr_condition == cond1) | (attr_condition == cond2)]
                    attr_runnum_temp = attr_runnum[(attr_condition == cond1) | (attr_condition == cond2)]
                    mean_acc = []
                   
                    for test_run in runs:

                        # all runs other than test_run
                        X_train = data_cond[attr_runnum_temp != test_run]
                        y_train = attr_condition_temp[attr_runnum_temp != test_run]

                        # test run data only
                        X_test = data_cond[attr_runnum_temp == test_run]
                        y_test = attr_condition_temp[attr_runnum_temp == test_run]

                        # fit
                        linear_SVM.fit(X_train, y_train)
                        #print(f'{roi} {cond} {subj} train: {linear_SVM.score(X_train, y_train)}')
                        #print(f'{roi} {cond} {subj} test: {linear_SVM.score(X_test, y_test)}')

                        mean_acc.append(linear_SVM.score(X_test, y_test))

                    mean_acc = np.mean(np.array(mean_acc)) # mean across runs

                    tmp_df = pd.DataFrame({'sub':[sub], 'roi':[roi], 'cond1':[cond1], 'cond2':[cond2], 'mean_acc':[mean_acc]})
                    all_sub_acc = pd.concat([all_sub_acc, tmp_df], ignore_index=True)

In [8]:
list_temp = np.array([None]*len(all_sub_acc))

ind_1 = all_sub_acc[(all_sub_acc.cond1==0) & (all_sub_acc.cond2==1)].index.values 
list_temp[ind_1.astype(int)] = 'WvsLH'

ind_2= all_sub_acc[(all_sub_acc.cond1==0) & (all_sub_acc.cond2==2)].index.values 
list_temp[ind_2.astype(int)] = 'WvsHH'

ind_3 = all_sub_acc[(all_sub_acc.cond1==1) & (all_sub_acc.cond2==2)].index.values 
list_temp[ind_3.astype(int)] = 'LHvsHH'

all_sub_acc['Comparison'] = list_temp
all_sub_acc['ROI_type'] = len(all_sub_acc)*['visual_rois']



In [ ]:
# Stats
# test significance

comparison = ['WvsLH', 'LHvsHH', 'WvsHH']
p_all_df = pd.DataFrame({'roi': [], 'Comparison': [], 'p-val': []})
for comp in comparison:
    
    dat_df_temp = all_sub_acc[all_sub_acc['Comparison'].str.contains(comp)]
    
    for roi in rois:
        dat_roi =  dat_df_temp[dat_df_temp['roi'].str.contains(roi)]
        stats_temp = pg.ttest(dat_roi['mean_acc'], 0.5, alternative='greater')
        p_temp = stats_temp['p-val'].values
        
        p_df_temp = pd.DataFrame({'roi': [roi], 'Comparison': [comp], 'p-val': p_temp})
        p_all_df = pd.concat([p_all_df, p_df_temp], ignore_index=True)

# correct p-values
p_all_corr = pd.DataFrame({'roi': [], 'Comparison': [], 'p-val': [], 'p-val_corr': []})
for comp in comparison:
    p_all_df_temp = p_all_df[p_all_df['Comparison'].str.contains(comp)]
    p_vals = p_all_df_temp['p-val'].values
    p_vals_corr = pg.multicomp(p_vals, alpha=0.05, method='fdr_bh')
    p_all_df_temp['p-val_corr'] = p_vals_corr[1]
    
    p_all_corr = pd.concat([p_all_corr, p_all_df_temp], ignore_index=False)
p_all_corr

In [ ]:
# Add asterisks

list_temp = np.array([None]*len(p_all_corr))

ind_1 = p_all_corr[p_all_corr['p-val_corr'] > 0.05].index.values 
list_temp[ind_1.astype(int)] = 'ns'

ind_2= p_all_corr[p_all_corr['p-val_corr'] <= 0.05].index.values 
list_temp[ind_2.astype(int)] = '*'

ind_2= p_all_corr[p_all_corr['p-val_corr'] <= 0.01].index.values 
list_temp[ind_2.astype(int)] = '**'

ind_3 = p_all_corr[p_all_corr['p-val_corr'] <= 0.001].index.values 
list_temp[ind_3.astype(int)] = '***'

ind_3 = p_all_corr[p_all_corr['p-val_corr'] <= 0.0001].index.values 
list_temp[ind_3.astype(int)] = '****'


p_all_corr['pval_asterisks'] = list_temp 
p_all_corr

In [ ]:
comp = 'LHvsHH' # 'WvsLH', 'LHvsHH', 'WvsHH'
save_fig = 0
asterisks =1 

roi_names = all_sub_acc['roi'].unique()
dat_df_temp = all_sub_acc[all_sub_acc['Comparison'].str.contains(comp)]
p_all_corr_temp = p_all_corr[p_all_corr['Comparison'].str.contains(comp)]

ax = sns.boxplot(x = 'roi',
                 y='mean_acc',
                 data=dat_df_temp, 
                 palette = 'Set2',
                 #hue= 'ROI_type',
                 dodge=False,
               #palette={'#EDEAE9', '#FECDA6'},
                 showcaps=False,
                 flierprops={"marker":""}) 
sns.stripplot(data=dat_df_temp, y="mean_acc", x = 'roi', 
              dodge=True,color={'black'},size=4,edgecolor='black') 
ax.set(ylabel='Accuracy')
y_position_title = all_sub_acc['mean_acc'].max()*1.3
ax.set_title(f'Visual regs | {comp}',y=y_position_title)  
ax.tick_params()
sns.despine()
ax.axhline(0.5, color='gray', linestyle='--')

# for asterisks
if asterisks:

    if comp == 'WvsLH':
        y_position = all_sub_acc['mean_acc'].max()*0.77
    elif comp == 'WvsHH':
        y_position = all_sub_acc['mean_acc'].max()*1.02
    elif comp == 'LHvsHH':
        y_position = all_sub_acc['mean_acc'].max()*0.97
        
    pval_asterisks = p_all_corr_temp['pval_asterisks'].values
    for idx, pval in enumerate(pval_asterisks):
        plt.text(x=idx, y=y_position, s=pval)

#plt.xticks(rotation=45)
plt.legend([],[],frameon=False)

fig = ax.get_figure()
#fig.set_figwidth(12)
if save_fig:
    if asterisks:
        fig.savefig(os.path.join(figdir, f'pain-induced-manifold-changes_heatlevel_classification_visualrois_{comp}_significance.pdf'), bbox_inches='tight')
    else:
        fig.savefig(os.path.join(figdir, f'pain-induced-manifold-changes_heatlevel_classification_visualrois_{comp}.pdf'), bbox_inches='tight')